# RR SSC-MEK PHASE B2: the control, then the scoring run
Pre-registration committed 2026-10-01 22:24 EDT, SHA-256 `45de35e26156bf1c70d22bc9100adaeec30f3fc4e70b803e075c124960c3a4c9`.
Frozen inputs from Phase B1, SHA-256 `c083e3cfb23cca67b26e772d8435b138d982db02bf74347ad13053c6c42e8301`. The notebook refuses any other freeze.

**Criterion: one condition, permutation p < 0.05 on the mean percentile of PD-0325901 across the surviving primary queries.** Lower percentile = stronger reversal. Ties take the midrank.

Order is fixed and enforced by the cells: the control of Section 9 runs and can void a query **before** any rank is printed.

Run only after B1 has finished in this folder. Run: hamburger > Runtime > Run all. Paste back everything between `=== SSC-MEK PHASE B2 REPORT BEGIN ===` and `=== SSC-MEK PHASE B2 REPORT END ===`. If a cell stops, paste the error instead.

In [ ]:
# CELL 1: load the frozen inputs and the library. Nothing below may change them.
from google.colab import drive
drive.mount("/content/drive")
import os, json, hashlib, time, collections, gzip, re, urllib.request, io
import numpy as np, pandas as pd
BENCH = "/content/drive/MyDrive/RitschelResearch/benchmark"
W = "/content/drive/MyDrive/RitschelResearch/ssc_mek"; os.chdir(W)
PREREG = "45de35e26156bf1c70d22bc9100adaeec30f3fc4e70b803e075c124960c3a4c9"
FREEZE = "c083e3cfb23cca67b26e772d8435b138d982db02bf74347ad13053c6c42e8301"
REPORT = []
def R(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); REPORT.append(s)
raw = open("sscmek_signatures_frozen_B1.json", "rb").read()
FROZEN_SHA = hashlib.sha256(raw).hexdigest()
assert FROZEN_SHA == FREEZE, ("NOT THE CHECKED B1 FREEZE", FROZEN_SHA)
FZ = json.loads(raw); SIG = FZ["signatures"]; DR = pd.DataFrame(FZ["drugs"])
assert FZ["prereg_sha256"] == PREREG, "FREEZE DOES NOT CARRY THE COMMITTED PRE-REGISTRATION HASH"
PRIM = FZ["primary_queries"]; SENS = FZ["sensitivity_queries"]
assert PRIM == ["Q1", "Q2"], ("PRIMARY QUERY SET CHANGED", PRIM)
g = pd.read_csv("geneinfo_beta.txt", sep="\t")
LMG = g.loc[g.feature_space == "landmark"].copy(); LMG["gene_id"] = LMG.gene_id.astype(int)
SYM2E = {str(s).upper(): int(e) for e, s in zip(LMG.gene_id, LMG.gene_symbol)}
ENS2E = {}
if "ensembl_id" in LMG.columns:
    for e, x in zip(LMG.gene_id, LMG.ensembl_id.astype(str)):
        for y in x.split("|"):
            if y.startswith("ENSG"): ENS2E[y.split(".")[0]] = int(e)
assert len(SYM2E) == 978, ("SYMBOL MAP IS NOT 978 LANDMARKS", len(SYM2E))
M = np.load(os.path.join(BENCH, "libA_matrix.npy"))
perts = np.load(os.path.join(BENCH, "libA_perts.npy"), allow_pickle=True)
genes = np.load(os.path.join(BENCH, "libA_genes.npy")); GI = {int(x): i for i, x in enumerate(genes)}
assert M.shape == (90834, 978) and len(perts) == 90834, M.shape
R("freeze", FROZEN_SHA)
R("prereg", PREREG)
R("library", M.shape, "compounds", len(set(perts.tolist())))
R("primary queries", PRIM, "| sensitivity", SENS)
R("gene maps rebuilt here: symbol", len(SYM2E), "ensembl", len(ENS2E))
R("drug table", len(DR), "IDs")

In [ ]:
# CELL 2: scoring code. ks/conn_all are copied VERBATIM from the deposited benchmark notebook.
# A vectorized form is added for speed and is accepted only after it reproduces conn_all exactly.
def ks(rp, n, t):
    if t == 0 or n == 0: return 0.0
    V = np.sort(np.asarray(rp, float)); j = np.arange(1, t + 1, dtype=float)
    a = np.max(j / t - V / n); b = np.max(V / n - (j - 1.0) / t)
    return float(a) if a > b else float(-b)
def conn_all(Mx, up, dn):
    n = Mx.shape[1]; out = np.empty(Mx.shape[0])
    order = np.argsort(-Mx, axis=1, kind="stable")
    rank = np.empty_like(order); ar = np.arange(1, n + 1)
    np.put_along_axis(rank, order, np.broadcast_to(ar, order.shape), axis=1)
    for i in range(Mx.shape[0]):
        eu = ks(rank[i, up], n, len(up)); ed = ks(rank[i, dn], n, len(dn))
        out[i] = 0.0 if eu * ed >= 0 else (eu - ed) / 2.0
    return out
def rank_matrix(Mx, chunk=8000):
    n = Mx.shape[1]; rk = np.empty(Mx.shape, dtype=np.int32); ar = np.arange(1, n + 1, dtype=np.int32)
    for a in range(0, Mx.shape[0], chunk):
        b = min(a + chunk, Mx.shape[0])
        o = np.argsort(-Mx[a:b], axis=1, kind="stable")
        np.put_along_axis(rk[a:b], o, np.broadcast_to(ar, o.shape), axis=1)
    return rk
def _ksv(sub, n, t):
    V = np.sort(sub, axis=1).astype(np.float64); j = np.arange(1, t + 1, dtype=np.float64)
    a = np.max(j / t - V / n, axis=1); b = np.max(V / n - (j - 1.0) / t, axis=1)
    return np.where(a > b, a, -b)
def conn_from_rank(rk, up, dn):
    n = rk.shape[1]
    eu = _ksv(rk[:, up], n, len(up)); ed = _ksv(rk[:, dn], n, len(dn))
    return np.where(eu * ed >= 0, 0.0, (eu - ed) / 2.0)
def midrank(v):
    v = np.asarray(v, float); n = v.size; o = np.argsort(v, kind="stable"); r = np.empty(n); i = 0
    while i < n:
        j = i
        while j + 1 < n and v[o[j + 1]] == v[o[i]]: j += 1
        r[o[i:j + 1]] = 0.5 * (i + j) + 1.0; i = j + 1
    return r
def spear(a, b):
    ra = midrank(a) - midrank(a).mean(); rb = midrank(b) - midrank(b).mean()
    d = np.sqrt((ra * ra).sum() * (rb * rb).sum())
    return float("nan") if d == 0 else float((ra * rb).sum() / d)
# POSITIVE CONTROL for the vectorized form: it must reproduce conn_all to the bit on a real subset.
_q = SIG[PRIM[0]]; _cols = [GI[e] for e in _q["usable"]]; _pos = {e: i for i, e in enumerate(_q["usable"])}
_up = np.array([_pos[e] for e in _q["up"]]); _dn = np.array([_pos[e] for e in _q["dn"]])
_sub = M[:3000][:, _cols]
_a = conn_all(_sub, _up, _dn); _b = conn_from_rank(rank_matrix(_sub), _up, _dn)
assert np.array_equal(_a, _b), ("VECTORIZED SCORE DIFFERS FROM THE DEPOSITED conn_all", float(np.abs(_a - _b).max()))
R("scoring check: the vectorized score reproduces the deposited conn_all exactly on 3,000 signatures")
R("  (nonzero scores in that subset: %d of 3000)" % int((_a != 0).sum()))

In [ ]:
# CELL 3: the B1 build code, copied verbatim so the rebuild cannot drift. SIG_REBUILT must equal the freeze.
# CELL 3: shared helpers. stat() is the benchmark's, unchanged.
UA = {"User-Agent": "Mozilla/5.0 (research; contact glen@ritschelresearch.com)"}
def fetch_to(u, path, timeout=900):
    if not os.path.exists(path):
        b = urllib.request.urlopen(urllib.request.Request(u, headers=UA), timeout=timeout).read()
        open(path, "wb").write(b)
    return path
def strip(x): return x.strip().strip('"').strip()
def series_meta(stem):
    f = stem + "_series_matrix.txt.gz"
    assert os.path.exists(f), "MISSING " + f + " (run Phase A first)"
    out = []
    with gzip.open(f, "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("!series_matrix_table_begin"): break
            out.append(line)
    return out
def fields_of(meta):
    acc = [l for l in meta if l.startswith("!Sample_geo_accession")]
    assert acc, "NO SAMPLE ACCESSION LINE"
    gsm = [strip(x) for x in acc[0].rstrip("\n").split("\t")[1:]]
    F_ = collections.OrderedDict()
    for l in meta:
        if l.startswith("!Sample_characteristics_ch1"):
            for i, v in enumerate(strip(x) for x in l.rstrip("\n").split("\t")[1:]):
                if ":" in v:
                    k, _, val = v.partition(":")
                    F_.setdefault(k.strip().lower(), {})[i] = val.strip()
    return gsm, F_
def stat(E, a, b, k=150):
    A = E[:, a]; B = E[:, b]; d = A.mean(1) - B.mean(1)
    pooled = np.sqrt(0.5 * (A.var(1, ddof=1) + B.var(1, ddof=1)))
    fl = np.median(pooled[pooled > 0]) if np.any(pooled > 0) else 1.0
    st = d / np.maximum(pooled, fl * 0.1)
    o = np.argsort(-st, kind="stable")
    return o[:k].copy(), o[-k:].copy()
SIG_REBUILT = {}
def build(qid, series, E, grp, note):
    keep = ~np.isnan(E).any(axis=1); idx = np.where(keep)[0]
    cases = [i for i, x in enumerate(grp) if x == "case"]
    ctrls = [i for i, x in enumerate(grp) if x == "control"]
    assert len(cases) > 0 and len(ctrls) > 0, "EMPTY ARM IS FATAL"
    assert int(keep.sum()) >= 300, ("TOO FEW USABLE LANDMARKS", int(keep.sum()))
    up, dn = stat(E[keep], cases, ctrls)
    SIG_REBUILT[qid] = dict(series=series, n_case=len(cases), n_ctrl=len(ctrls), genes_usable=int(keep.sum()),
                    up=[int(genes[idx[i]]) for i in up], dn=[int(genes[idx[i]]) for i in dn],
                    usable=[int(genes[i]) for i in idx], note=note)
    R("%s %s: cases %d controls %d usable landmarks %d | %s" % (qid, series, len(cases), len(ctrls), int(keep.sum()), note))
R("rebuild helpers ready: statistic is the benchmark's pooled-sd difference with a 0.1x median floor, k=150")
# CELL 4: platform annotation. A cached or downloaded file is NEVER trusted by name: it is accepted only
# if it actually contains a platform table. Sources are tried in order and every rejection is reported
# with the first characters of what came back, so a bad file is diagnosed rather than guessed at.
# (The first B1 run failed here: a file left by Phase A was accepted on its name alone.)
def _has_table(txt): return "!platform_table_begin" in txt
def platform_table(plat):
    cache = plat + "_table.txt"
    if os.path.exists(cache):
        t = open(cache, encoding="utf-8", errors="replace").read()
        if _has_table(t): R("platform", plat, "table from cache", cache, "(%d chars)" % len(t)); return t, "cache"
        os.remove(cache); R("platform", plat, "cached table rejected: no table marker; refetching")
    tries = []
    gz = plat + ".annot.gz"
    if os.path.exists(gz):
        try:
            t = gzip.open(gz, "rt", encoding="utf-8", errors="replace").read()
            if _has_table(t): open(cache, "w", encoding="utf-8").write(t); R("platform", plat, "from a local", gz); return t, "local annot.gz"
            tries.append("local %s: no table marker, %d chars, starts %r" % (gz, len(t), t[:120]))
        except Exception as e: tries.append("local %s: %s" % (gz, type(e).__name__))
        os.remove(gz); R("platform", plat, "removed the unusable local", gz)
    SRC = [("annot.gz", "https://ftp.ncbi.nlm.nih.gov/geo/platforms/%snnn/%s/annot/%s.annot.gz" % (plat[:-3], plat, plat)),
           ("acc.cgi full", "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=%s&targ=self&form=text&view=full" % plat),
           ("acc.cgi data", "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=%s&targ=self&form=text&view=data" % plat)]
    for label, u in SRC:
        try:
            raw = urllib.request.urlopen(urllib.request.Request(u, headers=UA), timeout=900).read()
            t = gzip.decompress(raw).decode("utf-8", "replace") if raw[:2] == b"\x1f\x8b" else raw.decode("utf-8", "replace")
            if _has_table(t):
                open(cache, "w", encoding="utf-8").write(t)
                R("platform", plat, "table from", label, "(%d chars)" % len(t)); return t, label
            tries.append("%s: no table marker, %d chars, starts %r" % (label, len(t), t[:120]))
        except Exception as e:
            tries.append("%s: %s %s" % (label, type(e).__name__, str(e)[:80]))
        R("platform", plat, "rejected", label, "->", tries[-1])
    raise AssertionError(("NO PLATFORM TABLE FOR " + plat, tries))
def probe_to_entrez(plat):
    txt, src = platform_table(plat)
    lines = txt.splitlines(); i = 0
    while i < len(lines) and not lines[i].startswith("!platform_table_begin"): i += 1
    hdr = [c.strip() for c in lines[i + 1].split("\t")]
    cand_e = [j for j, c in enumerate(hdr) if c.upper() in ("GENE ID", "GENE", "ENTREZ_GENE_ID", "ENTREZ GENE ID", "GENE_ID", "LOCUSLINK_ID")]
    cand_s = [j for j, c in enumerate(hdr) if c.upper() in ("GENE SYMBOL", "GENE_SYMBOL", "SYMBOL", "ORF")]
    R("platform", plat, "columns:", hdr[:14], "| entrez col", [hdr[j] for j in cand_e], "| symbol col", [hdr[j] for j in cand_s])
    assert cand_e or cand_s, ("NO GENE COLUMN ON " + plat, hdr)
    m = {}; via = collections.Counter()
    for ln in lines[i + 2:]:
        if ln.startswith("!platform_table_end"): break
        q = ln.rstrip("\n").split("\t")
        if not q or not q[0]: continue
        e = None
        for j in cand_e:
            if len(q) > j:
                x = q[j].split("///")[0].strip()
                if x.isdigit(): e = int(x); via["entrez"] += 1; break
        if e is None:
            for j in cand_s:
                if len(q) > j:
                    e = SYM2E.get(q[j].split("///")[0].strip().upper())
                    if e: via["symbol"] += 1; break
        if e in GI: m[q[0].strip()] = e
    R("platform", plat, "source", src, "| probes mapped to a landmark:", len(m), "| distinct landmarks:", len(set(m.values())), "of 978 | by", dict(via))
    PLATSRC[plat] = src
    return m
PLATSRC = {}
def matrix_values(stem, gsm):
    rows = []; hdr = None; inb = False
    with gzip.open(stem + "_series_matrix.txt.gz", "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("!series_matrix_table_end"): break
            if inb:
                q = line.rstrip("\n").split("\t")
                if hdr is None: hdr = [strip(x) for x in q]
                else: rows.append(q)
                continue
            if line.startswith("!series_matrix_table_begin"): inb = True
    assert hdr is not None and hdr[1:] == gsm, "MATRIX COLUMNS DO NOT MATCH THE SAMPLE ACCESSIONS"
    return rows
def collapse(stem, gsm, plat):
    p2e = probe_to_entrez(plat)
    rows = matrix_values(stem, gsm)
    acc = {}
    for q in rows:
        e = p2e.get(strip(q[0]))
        if e is None: continue
        try: v = np.array([float(strip(x)) if strip(x) not in ("", "NA", "null", "NaN") else np.nan for x in q[1:]])
        except ValueError: continue
        acc.setdefault(GI[e], []).append(v)
    E = np.full((978, len(gsm)), np.nan)
    for gi_, L in acc.items(): E[gi_] = np.nanmedian(np.vstack(L), axis=0)
    fin = np.isfinite(E).all(axis=1)
    assert fin.any(), ("NO LANDMARK HAS VALUES IN EVERY SAMPLE FOR " + stem)
    p99 = np.nanpercentile(E[fin], 99)
    R(stem, "collapsed: rows in table", len(rows), "| landmarks with values", int(fin.sum()), "| 99th percentile %.3f" % p99)
    if p99 > 100:
        E = np.log2(np.maximum(E, 1)); R(stem, "99th percentile exceeds 100 -> log2(max(x,1)) applied [pre-registered scale rule]")
    else:
        R(stem, "99th percentile at or below 100 -> used as deposited [pre-registered scale rule]")
    return E

In [ ]:
# CELL 4: rebuild each query's expression matrix and PROVE the rebuild reproduces the freeze exactly.
# The control of Section 9 needs per-sample expression, which the freeze does not carry. Rebuilding it
# is only safe if it is shown to give the frozen signature back, gene for gene.
EXPR = {}
def q1_rebuild():
    stem = "GSE181549"; meta = series_meta(stem); gsm, FF = fields_of(meta)
    cond = FF["condition"]; num = FF["sample number within subject"]
    sel = [i for i in range(len(gsm)) if num.get(i) == "1"]
    grp = ["case" if cond.get(i) == "systemic sclerosis patient" else "control" for i in sel]
    E = collapse(stem, gsm, "GPL13497")
    return stem, E[:, sel], grp, "one biopsy per subject (sample number 1); groups from the condition field"
def q2_rebuild():
    stem = "GSE130955"; meta = series_meta(stem); gsm, FF = fields_of(meta)
    supp = [[strip(x) for x in l.rstrip("\n").split("\t")[1:]] for l in meta if l.startswith("!Sample_supplementary_file")]
    flat = [x if x and x.upper() != "NONE" else "" for x in supp[0]]
    tabs = {}; ints = 0; tot = 0
    for i, x in enumerate(flat):
        n = x.split("/")[-1]
        p = fetch_to("https://ftp.ncbi.nlm.nih.gov/geo/samples/%snnn/%s/suppl/%s" % (gsm[i][:-3], gsm[i], n), n, timeout=300)
        d = {}
        with (gzip.open if n.endswith(".gz") else open)(p, "rt", encoding="utf-8", errors="replace") as fh:
            for ln in fh:
                q = ln.rstrip("\n").split("\t")
                if len(q) < 2: continue
                try: v = float(q[-1])
                except ValueError: continue
                d[q[0].strip()] = v
                if i == 0: tot += 1; ints += (v == int(v))
        tabs[gsm[i]] = d
    R("Q2 deposit check: of %d values in the first file, %d are whole numbers (%.1f%%)" % (tot, ints, 100.0 * ints / max(tot, 1)))
    ids = sorted(set().union(*[set(d) for d in tabs.values()]))
    Mx = np.array([[tabs[s].get(k, 0.0) for s in gsm] for k in ids], dtype=float)
    lib = Mx.sum(0)
    R("Q2 per-sample sum of all %d genes: min %.0f median %.0f max %.0f" % (len(ids), lib.min(), np.median(lib), lib.max()))
    R("Q2 NOTE: a raw-count library is 1e7 or more and a TPM column sums to exactly 1e6. These sums sit near")
    R("   1e6 and vary, which is what a per-kilobase normalized deposit (FPKM/RPKM) looks like. The pre-registered")
    R("   transformation is applied unchanged; the write-up must not call these counts.")
    CPM = np.log2(Mx / np.maximum(lib, 1) * 1e6 + 1)
    acc = {}
    for i, k in enumerate(ids):
        e = ENS2E.get(k.split(".")[0]) or SYM2E.get(k.strip().upper())
        if e in GI: acc.setdefault(GI[e], []).append(CPM[i])
    E = np.full((978, len(gsm)), np.nan)
    for gi_, L in acc.items(): E[gi_] = np.median(np.vstack(L), axis=0)
    dx = FF["diagnosis"]; bn = FF["biopsy number for this subject"]
    sel = [i for i in range(len(gsm)) if bn.get(i) == "1"]
    grp = ["case" if dx.get(i) == "SSc patient" else "control" for i in sel]
    return stem, E[:, sel], grp, "first biopsy per subject; log2 CPM, library size on the full matrix"
def s1_rebuild():
    stem = "GSE95065"; meta = series_meta(stem); gsm, FF = fields_of(meta)
    dx = FF["dx"]
    grp = ["case" if dx.get(i) == "diffuse cutaneous systemic sclerosis" else "control" for i in range(len(gsm))]
    E = collapse(stem, gsm, "GPL23080")
    return stem, E, grp, "sensitivity only; same laboratory as the P01 source data, so individuals may overlap"
for qid, fn in [("Q1", q1_rebuild), ("Q2", q2_rebuild), ("S1", s1_rebuild)]:
    if qid not in SIG: R("skipping", qid, "- not in the freeze"); continue
    stem, E, grp, note = fn()
    build(qid, stem, E, grp, note)
    a, b = SIG[qid], SIG_REBUILT[qid]
    assert a["up"] == b["up"] and a["dn"] == b["dn"] and a["usable"] == b["usable"], ("REBUILD DOES NOT REPRODUCE THE FREEZE", qid)
    assert a["n_case"] == b["n_case"] and a["n_ctrl"] == b["n_ctrl"], ("REBUILD ARM COUNTS DIFFER", qid)
    keep = ~np.isnan(E).any(axis=1)
    EXPR[qid] = dict(E=E[keep], grp=grp)
    R("rebuild of", qid, "reproduces the freeze exactly: 150 up, 150 dn,", len(a["usable"]), "usable genes")
assert set(EXPR) >= set(PRIM), ("A PRIMARY QUERY DID NOT REBUILD", sorted(EXPR))

In [ ]:
# CELL 5: THE CONTROL OF SECTION 9, run before any rank is printed. Healthy controls are split at random
# into two halves 20 times; each split is scored as a query and its library ranking is correlated with the
# ranking from the real disease signature. Median rho above 0.5 VOIDS that query.
VOID = {}; RHO = {}
for qid in sorted(EXPR):
    s = SIG[qid]; cols = [GI[e] for e in s["usable"]]; pos = {e: i for i, e in enumerate(s["usable"])}
    up = np.array([pos[e] for e in s["up"]]); dn = np.array([pos[e] for e in s["dn"]])
    rk = rank_matrix(M[:, cols])
    real = conn_from_rank(rk, up, dn)
    real_c = pd.DataFrame({"p": perts, "s": real}).groupby("p")["s"].median()
    E = EXPR[qid]["E"]; grp = EXPR[qid]["grp"]
    ctrl = [i for i, x in enumerate(grp) if x == "control"]
    rhos = []; undef = 0
    for i in range(20):
        rng = np.random.default_rng(12345 + i)
        perm = rng.permutation(ctrl); h = len(perm) // 2
        a, b = list(perm[:h]), list(perm[h:])
        u2, d2 = stat(E, a, b)
        sc = conn_from_rank(rk, u2, d2)
        comp = pd.DataFrame({"p": perts, "s": sc}).groupby("p")["s"].median()
        r = spear(comp.values, real_c.reindex(comp.index).values)
        if np.isnan(r): undef += 1
        else: rhos.append(r)
    med = float(np.median(rhos)) if rhos else float("nan")
    RHO[qid] = dict(median=med, n=len(rhos), undefined=undef,
                    lo=float(min(rhos)) if rhos else None, hi=float(max(rhos)) if rhos else None)
    inconclusive = undef > 10
    VOID[qid] = bool(inconclusive or (rhos and med > 0.5))
    R("control %s: %d controls split %d/%d, 20 splits | median rho %.4f | range %.3f to %.3f | undefined %d | %s"
      % (qid, len(ctrl), len(ctrl) // 2, len(ctrl) - len(ctrl) // 2, med,
         RHO[qid]["lo"] if rhos else float("nan"), RHO[qid]["hi"] if rhos else float("nan"), undef,
         "INCONCLUSIVE, NOT REPORTED" if inconclusive else ("VOID, RANKS NOT REPORTED" if VOID[qid] else "not void")))
LIVE = [q for q in PRIM if not VOID.get(q, True)]
R("primary queries surviving the control:", LIVE, "of", PRIM)
assert LIVE, "EVERY PRIMARY QUERY IS VOID. NO RANKS ARE REPORTED. STOP."

In [ ]:
# CELL 6: score the surviving queries. Compound score = median over its signatures. Percentile =
# (midrank - 0.5)/N, ascending, so the strongest reversal is lowest.
PCT = {}; TIE = {}; NC = {}; ZERO = {}; t0 = time.time()
for qid in sorted(EXPR):
    if VOID.get(qid, True) and qid in PRIM: R("skipping", qid, "- voided by the control"); continue
    s = SIG[qid]; cols = [GI[e] for e in s["usable"]]; pos = {e: i for i, e in enumerate(s["usable"])}
    up = np.array([pos[e] for e in s["up"]]); dn = np.array([pos[e] for e in s["dn"]])
    assert len(up) == 150 and len(dn) == 150 and not set(up.tolist()) & set(dn.tolist()), ("QUERY SETS MALFORMED", qid)
    sc = conn_from_rank(rank_matrix(M[:, cols]), up, dn)
    comp = pd.DataFrame({"p": perts, "s": sc}).groupby("p")["s"].median()
    PCT[qid] = pd.Series((midrank(comp.values) - 0.5) / len(comp), index=comp.index)
    TIE[qid] = float((comp == 0).mean()); NC[qid] = int(len(comp))
    z = comp.index[comp.values == 0.0]
    ZERO[qid] = float(PCT[qid][z[0]]) if len(z) else None
    R("scored %s: compounds %d | exact-zero fraction %.3f | the tied block sits at percentile %s | %ds"
      % (qid, NC[qid], TIE[qid], "none" if ZERO[qid] is None else round(ZERO[qid], 4), round(time.time() - t0)))
PD_IDS = sorted(DR[DR.role == "primary"].pert_id)
assert len(PD_IDS) == 2, ("PD-0325901 SHOULD HAVE TWO IDS", PD_IDS)
for qid in PCT:
    miss = [p for p in PD_IDS if p not in PCT[qid].index]
    assert not miss, ("PD-0325901 ID NOT SCORED", qid, miss)

In [ ]:
# CELL 7: the pre-registered primary. Statistic = the mean, over the surviving primary queries, of
# PD-0325901's percentile (the median of its two IDs). Null = 10,000 draws of two distinct library
# identifiers, the SAME two in every query, scored by the same median-then-mean structure.
def drug_stat(ids, queries):
    return float(np.mean([float(np.median(PCT[q].reindex(ids).values)) for q in queries]))
PER_Q = {q: float(np.median(PCT[q].reindex(PD_IDS).values)) for q in LIVE}
STAT = float(np.mean([PER_Q[q] for q in LIVE]))
for q in LIVE:
    R("PD-0325901 in %s: %s -> %s | drug percentile %.4f"
      % (q, PD_IDS, [round(float(PCT[q][p]), 4) for p in PD_IDS], PER_Q[q]))
R("OBSERVED statistic (mean over %s): %.4f" % (LIVE, STAT))
B = 10000; rng = np.random.default_rng(20261001)
IDX = {q: PCT[q].values for q in LIVE}
common = sorted(set.intersection(*[set(PCT[q].index) for q in LIVE]))
POSN = {q: {p: i for i, p in enumerate(PCT[q].index)} for q in LIVE}
cidx = {q: np.array([POSN[q][p] for p in common]) for q in LIVE}
R("null draws from the %d compounds scored in every surviving query" % len(common))
NULL = np.empty(B); t0 = time.time()
for b in range(B):
    pick = rng.choice(len(common), size=2, replace=False)
    NULL[b] = np.mean([np.median(IDX[q][cidx[q][pick]]) for q in LIVE])
p = (1 + int((NULL <= STAT).sum())) / (B + 1)
R("NULL mean %.4f sd %.4f over %d draws (%ds)" % (NULL.mean(), NULL.std(), B, round(time.time() - t0)))
R("PERMUTATION p = %.5f" % p)
R("CRITERION (one condition, p < 0.05): %s" % ("MET" if p < 0.05 else "NOT MET"))

In [ ]:
# CELL 8: everything the pre-registration requires to be reported whatever the outcome.
DETAIL = []
for qid in sorted(PCT):
    for _, r in DR.iterrows():
        if r.pert_id in PCT[qid].index:
            on = (ZERO[qid] is not None and abs(float(PCT[qid][r.pert_id]) - ZERO[qid]) < 1e-12)
            DETAIL.append((qid, r.role, r.drug, r.pert_id, float(PCT[qid][r.pert_id]), bool(on)))
R("--- every identifier in every scored query (percentile; lower = stronger reversal) ---")
for qid, role, drug, pid, v, on in DETAIL:
    R("  %-3s %-9s %-22s %s %.4f%s" % (qid, role, drug, pid, v, "  ON THE TIED BLOCK" if on else ""))
R("--- the tied block ---")
for qid in sorted(PCT):
    blk = [(role, d, p) for q2, role, d, p, v, on in DETAIL if q2 == qid and on]
    R("  %s: exact-zero fraction %.3f of %d compounds | block at percentile %s" %
      (qid, TIE[qid], NC[qid], "none" if ZERO[qid] is None else round(ZERO[qid], 4)))
    R("     identifiers on it: %s" % (blk if blk else "none"))
R("--- declared secondary: the MEK1/2 class (11 drugs in 14 identifiers), reported, never gating ---")
CLS = DR[DR.role.isin(["primary", "class"])]
sizes = [len(g) for d, g in CLS.groupby("drug")]
for qid in sorted(PCT):
    per = [float(np.median(PCT[qid].reindex(list(g.pert_id)).values)) for d, g in CLS.groupby("drug")]
    R("  %s: median over the 11 drugs %.4f" % (qid, float(np.median(per))))
CLS_STAT = float(np.mean([float(np.median([float(np.median(PCT[q].reindex(list(g.pert_id)).values)) for d, g in CLS.groupby("drug")])) for q in LIVE]))
rng2 = np.random.default_rng(777); CN = np.empty(2000)
for b in range(2000):
    tot = []
    for q in LIVE:
        pick = rng2.choice(len(common), size=sum(sizes), replace=False); i = 0; per = []
        for k in sizes:
            per.append(np.median(IDX[q][cidx[q][pick[i:i + k]]])); i += k
        tot.append(np.median(per))
    CN[b] = np.mean(tot)
cls_p = (1 + int((CN <= CLS_STAT).sum())) / 2001
R("  class statistic over %s: %.4f | null mean %.4f sd %.4f | p = %.4f (2,000 draws)" % (LIVE, CLS_STAT, CN.mean(), CN.std(), cls_p))
R("--- declared sensitivity query ---")
if "S1" in PCT:
    R("  S1 GSE95065: PD-0325901 %.4f | exact-zero %.3f. Reported only: it shares a laboratory with the"
      % (float(np.median(PCT["S1"].reindex(PD_IDS).values)), TIE["S1"]))
    R("     P01 source data, so individuals may overlap, and it never enters the primary.")
else:
    R("  S1 not scored")
R("--- reference compounds, reported only ---")
for qid in sorted(PCT):
    for _, r in DR[DR.role == "reference"].iterrows():
        if r.pert_id in PCT[qid].index: R("  %s %-22s %.4f" % (qid, r.drug, float(PCT[qid][r.pert_id])))
R("--- the mesenchymal-only (ASC) secondary is Phase B3, not this notebook ---")

In [ ]:
# CELL 9: freeze the result and print the report.
out = dict(prereg_sha256=PREREG, frozen_sha256=FROZEN_SHA, primary_queries=PRIM, live_queries=LIVE,
           control_rho=RHO, void=VOID, observed=STAT, per_query=PER_Q, p=p, B=B,
           null_mean=float(NULL.mean()), null_sd=float(NULL.std()), tie_frac=TIE, tie_block=ZERO,
           n_compounds=NC, class_stat=CLS_STAT, class_p=cls_p, detail=DETAIL)
bts = json.dumps(out, sort_keys=True, default=float).encode()
open("sscmek_B2_result.json", "wb").write(bts)
assert open("sscmek_B2_result.json", "rb").read() == bts, "RESULT READ-BACK MISMATCH"
np.save("sscmek_B2_null.npy", NULL)
print("=== SSC-MEK PHASE B2 REPORT BEGIN ===")
print("\n".join(REPORT))
print("--- HEADLINE ---")
print("pre-registration", PREREG)
print("scored against sscmek_signatures_frozen_B1.json sha256", FROZEN_SHA)
print("surviving primary queries", LIVE, "of", PRIM)
for q in LIVE: print("  PD-0325901 percentile in %s: %.4f" % (q, PER_Q[q]))
print("OBSERVED MEAN PERCENTILE %.4f" % STAT)
print("NULL mean %.4f sd %.4f (%d draws of two library identifiers)" % (NULL.mean(), NULL.std(), B))
print("PERMUTATION p = %.5f" % p)
print("CRITERION (p < 0.05):", "MET" if p < 0.05 else "NOT MET")
print("sscmek_B2_result.json sha256", hashlib.sha256(bts).hexdigest())
print("=== SSC-MEK PHASE B2 REPORT END ===")